# HYDE Query Design — `app/db/hyde.py`

Step-by-step development and benchmarking of the SQL queries that will back
`app/db/hyde.py`. Each cell is a self-contained step; run them in order.

**Goals**
- Verify `temporal.hyde_cells` and `temporal.hyde_times` are loaded correctly
- Resolve L8 and L6 basin ids for the Timbuktu reference site
- Benchmark cell-count and full-aggregation queries at both levels
- Draft the polygon-based query (Cliopatria proxy) using an L6 basin geometry
- Confirm response shape that `hyde.py` will return

In [1]:
# Cell 1 — imports and connection
import sys, time
from pathlib import Path
import pandas as pd

ROOT = Path('../../..')
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect

conn = db_connect()
print('Connected:', conn.execute('SELECT current_database(), inet_server_port()').fetchone())

Connected: ('cedop', 5435)


In [2]:
# Cell 2 — verify table row counts and time axis
print('hyde_cells :', conn.execute('SELECT count(*) FROM temporal.hyde_cells').fetchone()[0])
print('hyde_times :', conn.execute('SELECT count(*) FROM temporal.hyde_times').fetchone()[0])

times = pd.DataFrame(
    conn.execute('SELECT step_idx, year_ce FROM temporal.hyde_times ORDER BY step_idx').fetchall(),
    columns=['step_idx', 'year_ce']
)
print()
print('First 10 time steps:')
print(times.head(10).to_string(index=False))
print('...')
print('Last 10 time steps:')
print(times.tail(10).to_string(index=False))

hyde_cells : 2215829
hyde_times : 128

First 10 time steps:
 step_idx  year_ce
        0   -10000
        1    -9000
        2    -8000
        3    -7000
        4    -6000
        5    -5000
        6    -4000
        7    -3000
        8    -2000
        9    -1000
...
Last 10 time steps:
 step_idx  year_ce
      118     2016
      119     2017
      120     2018
      121     2019
      122     2020
      123     2021
      124     2022
      125     2023
      126     2024
      127     2025


In [3]:
# Cell 3 — resolve Timbuktu L8 and L6 basin ids

LAT, LON = 16.76618535, -3.00777252   # Timbuktu

row8 = conn.execute("""
    SELECT hybas_id, sub_area, up_area
    FROM public.basin08
    WHERE ST_Covers(geom, ST_SetSRID(ST_MakePoint(%s, %s), 4326))
    ORDER BY sub_area ASC
    LIMIT 1
""", (LON, LAT)).fetchone()
hybas_l8, sub_l8, up_l8 = row8
print(f'L8  hybas_id={hybas_l8}  sub_area={sub_l8:.1f} km²  up_area={up_l8:.1f} km²')

row6 = conn.execute("""
    SELECT hybas_id, sub_area, up_area
    FROM public.basin06
    WHERE ST_Covers(geom, ST_SetSRID(ST_MakePoint(%s, %s), 4326))
    ORDER BY sub_area ASC
    LIMIT 1
""", (LON, LAT)).fetchone()
hybas_l6, sub_l6, up_l6 = row6
print(f'L6  hybas_id={hybas_l6}  sub_area={sub_l6:.1f} km²  up_area={up_l6:.1f} km²')

L8  hybas_id=1080563570.0  sub_area=587.8 km²  up_area=588.1 km²
L6  hybas_id=1060551560.0  sub_area=3826.4 km²  up_area=382644.3 km²


In [4]:
# Cell 4 — confirm basin sizes and set geom_table_l6 for downstream cells

geom_table_l6 = 'public.basin06'

print(f'L8 sub_area : {sub_l8:.1f} km²')
print(f'L6 sub_area : {sub_l6:.1f} km²')
print(f'L6/L8 ratio : {sub_l6/sub_l8:.0f}×  (how many L8 cells fit in one L6 basin)')
print(f'geom_table_l6 = {geom_table_l6}')

L8 sub_area : 587.8 km²
L6 sub_area : 3826.4 km²
L6/L8 ratio : 7×  (how many L8 cells fit in one L6 basin)
geom_table_l6 = public.basin06


In [5]:
# Cell 5 — how many HYDE cells fall in each basin?
# ST_Within(ST_Centroid(cell.geom), basin.geom) = polygon-interior method (F9.1).

def count_cells(hybas_id, table):
    t0 = time.perf_counter()
    n = conn.execute(f"""
        SELECT COUNT(*)
        FROM temporal.hyde_cells hc
        JOIN {table} b ON b.hybas_id = %s
        WHERE ST_Within(ST_Centroid(hc.geom), b.geom)
    """, (hybas_id,)).fetchone()[0]
    elapsed = time.perf_counter() - t0
    return n, elapsed

n_l8, t_l8 = count_cells(hybas_l8, 'public.basin08')
print(f'L8 basin ({hybas_l8}): {n_l8} cells  ({t_l8*1000:.0f} ms)')

n_l6, t_l6 = count_cells(hybas_l6, geom_table_l6)
print(f'L6 basin ({hybas_l6}): {n_l6} cells  ({t_l6*1000:.0f} ms)')

L6 basin (1060551560.0): 45 cells  (5205 ms)


In [6]:
# Cell 6 — how many HYDE time steps fall in typical query windows?
# Helps set expectations for the cross-join size.

windows = [
    ('narrow BCE',   -1100, -900),
    ('narrow CE',    1000, 1200),
    ('medium CE',     500, 1500),
    ('broad CE',        0, 1900),
    ('full range', -10000, 2025),
]

print(f'{"Window":<20} {"from":>8} {"to":>8} {"n_steps":>8}  years')
print('-' * 70)
for label, fy, ty in windows:
    rows = conn.execute("""
        SELECT step_idx, year_ce FROM temporal.hyde_times
        WHERE year_ce BETWEEN %s AND %s ORDER BY year_ce
    """, (fy, ty)).fetchall()
    years = [r[1] for r in rows]
    print(f'{label:<20} {fy:>8} {ty:>8} {len(rows):>8}  {years}')

Window                   from       to  n_steps  years
----------------------------------------------------------------------
narrow BCE              -1100     -900        1  [-1000]
narrow CE                1000     1200        3  [1000, 1100, 1200]
medium CE                 500     1500       11  [500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500]
broad CE                    0     1900       38  [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1710, 1720, 1730, 1740, 1750, 1760, 1770, 1780, 1790, 1800, 1810, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900]
full range             -10000     2025      128  [-10000, -9000, -8000, -7000, -6000, -5000, -4000, -3000, -2000, -1000, 0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1710, 1720, 1730, 1740, 1750, 1760, 1770, 1780, 1790, 1800, 1810, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1951,

In [7]:
# Cell 7 — full aggregation query, L8 basin, narrow window (1000–1200 CE)
# This is the baseline: the query hyde.py will run for a typical signature call.

FROM_YEAR, TO_YEAR = 1000, 1200

AGG_SQL = """
WITH basin_cells AS (
    SELECT hc.cropland, hc.grazing, hc.pasture, hc.rangeland, hc.area_km2
    FROM temporal.hyde_cells hc
    JOIN public.basin08 b ON b.hybas_id = %(hybas_id)s
    WHERE ST_Within(ST_Centroid(hc.geom), b.geom)
),
steps AS (
    SELECT step_idx, year_ce
    FROM temporal.hyde_times
    WHERE year_ce BETWEEN %(from_year)s AND %(to_year)s
)
SELECT
    s.year_ce,
    round(SUM(bc.cropland [s.step_idx + 1])::numeric, 3) AS cropland_km2,
    round(SUM(bc.grazing  [s.step_idx + 1])::numeric, 3) AS grazing_km2,
    round(SUM(bc.pasture  [s.step_idx + 1])::numeric, 3) AS pasture_km2,
    round(SUM(bc.rangeland[s.step_idx + 1])::numeric, 3) AS rangeland_km2,
    round(SUM(bc.area_km2)::numeric, 1)                  AS basin_area_km2,
    COUNT(*)                                              AS n_cells
FROM basin_cells bc
CROSS JOIN steps s
GROUP BY s.year_ce
ORDER BY s.year_ce;
"""

t0 = time.perf_counter()
rows = conn.execute(AGG_SQL, {'hybas_id': hybas_l8, 'from_year': FROM_YEAR, 'to_year': TO_YEAR}).fetchall()
elapsed = time.perf_counter() - t0

cols = ['year_ce', 'cropland_km2', 'grazing_km2', 'pasture_km2', 'rangeland_km2', 'basin_area_km2', 'n_cells']
df = pd.DataFrame(rows, columns=cols)
print(f'L8 narrow window ({FROM_YEAR}–{TO_YEAR}): {len(df)} epochs, {elapsed*1000:.0f} ms')
print(df.to_string(index=False))

L8 narrow window (1000–1200): 3 epochs, 6771 ms
 year_ce cropland_km2 grazing_km2 pasture_km2 rangeland_km2 basin_area_km2  n_cells
    1000        0.079       4.705       0.000         4.705          573.4        7
    1100        0.085       4.905       0.000         4.905          573.4        7
    1200        0.090       5.083       0.000         5.083          573.4        7


In [8]:
# Cell 7b — diagnose: confirm the GIST index on geom is not helping ST_Centroid predicate,
# then create a functional index on ST_Centroid(geom) and retest.
#
# The predicate ST_Within(ST_Centroid(hc.geom), b.geom) wraps the indexed column in a
# function, so PostgreSQL cannot use the GIST index on geom — it falls back to a full
# 2.2M-row seq scan.  A functional index on ST_Centroid(geom) fixes this.

# Show query plan before index (should show Seq Scan or poor index use)
plan = conn.execute("""
    EXPLAIN (FORMAT TEXT)
    SELECT COUNT(*)
    FROM temporal.hyde_cells hc
    JOIN public.basin08 b ON b.hybas_id = %s
    WHERE ST_Within(ST_Centroid(hc.geom), b.geom)
""", (hybas_l8,)).fetchall()
print('--- PLAN BEFORE INDEX ---')
for row in plan:
    print(row[0])

# Create the functional index (one-time; IF NOT EXISTS guards re-runs)
print('\nCreating functional index on ST_Centroid(geom)...')
t0 = time.perf_counter()
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_hyde_cells_centroid
    ON temporal.hyde_cells USING GIST (ST_Centroid(geom))
""")
conn.commit()
print(f'Done ({(time.perf_counter()-t0):.1f}s)')

# Show plan after index
plan2 = conn.execute("""
    EXPLAIN (FORMAT TEXT)
    SELECT COUNT(*)
    FROM temporal.hyde_cells hc
    JOIN public.basin08 b ON b.hybas_id = %s
    WHERE ST_Within(ST_Centroid(hc.geom), b.geom)
""", (hybas_l8,)).fetchall()
print('\n--- PLAN AFTER INDEX ---')
for row in plan2:
    print(row[0])

Done (7.9s)

--- PLAN AFTER INDEX ---
Aggregate  (cost=327576.15..327576.16 rows=1 width=8)
  ->  Nested Loop  (cost=0.41..327570.61 rows=2216 width=0)
        ->  Seq Scan on basin08 b  (cost=0.00..260455.20 rows=1 width=2999)
              Filter: (hybas_id = '1080563570'::double precision)
        ->  Index Scan using idx_hyde_cells_centroid on hyde_cells hc  (cost=0.41..67113.19 rows=222 width=120)
              Index Cond: (st_centroid(geom) @ b.geom)
              Filter: st_within(st_centroid(geom), b.geom)


In [9]:
# Cell 7c — rerun narrow-window query after functional index; expect large speedup

t0 = time.perf_counter()
rows = conn.execute(AGG_SQL, {'hybas_id': hybas_l8, 'from_year': 1000, 'to_year': 1200}).fetchall()
elapsed = time.perf_counter() - t0

df = pd.DataFrame(rows, columns=cols)
print(f'L8 narrow window (1000–1200) WITH index: {len(df)} epochs, {elapsed*1000:.0f} ms')
print(df.to_string(index=False))

L8 narrow window (1000–1200) WITH index: 3 epochs, 1128 ms
 year_ce cropland_km2 grazing_km2 pasture_km2 rangeland_km2 basin_area_km2  n_cells
    1000        0.079       4.705       0.000         4.705          573.4        7
    1100        0.085       4.905       0.000         4.905          573.4        7
    1200        0.090       5.083       0.000         5.083          573.4        7


In [10]:
# Cell 8b — try two faster spatial predicates and compare timing
#
# Current: ST_Within(ST_Centroid(hc.geom), b.geom)  — functional index, 1128ms
# Try A:   ST_Intersects(hc.geom, b.geom)            — uses existing GIST on hc.geom directly
# Try B:   ST_Within(hc.geom, b.geom)                — cell fully inside basin; uses hc.geom GIST
#
# Run ANALYZE first to freshen planner statistics after the new index.

conn.execute('ANALYZE temporal.hyde_cells')
conn.commit()
print('ANALYZE done')

AGG_INTERSECTS = AGG_SQL.replace(
    'ST_Within(ST_Centroid(hc.geom), b.geom)',
    'ST_Intersects(hc.geom, b.geom)'
)
AGG_WITHIN = AGG_SQL.replace(
    'ST_Within(ST_Centroid(hc.geom), b.geom)',
    'ST_Within(hc.geom, b.geom)'
)

for label, sql in [
    ('ST_Centroid + functional index (current)', AGG_SQL),
    ('ST_Intersects on hc.geom',                 AGG_INTERSECTS),
    ('ST_Within(hc.geom, b.geom)',               AGG_WITHIN),
]:
    t0 = time.perf_counter()
    rows = conn.execute(sql, {'hybas_id': hybas_l8, 'from_year': 1000, 'to_year': 1200}).fetchall()
    ms = (time.perf_counter() - t0) * 1000
    print(f'{label:<45}  {len(rows)} epochs  {ms:.0f} ms  ({rows[0][6]} cells)')

ST_Within(hc.geom, b.geom)                     3 epochs  590 ms  (1 cells)


In [ ]:
# Cell 11 — response shape: what hyde.py will return
#
# psycopg3 returns SQL numeric/decimal as Python Decimal objects.
# Cast the whole df to float64 up front; int columns get re-cast explicitly.

def rows_to_response(df):
    """Convert aggregation result to the API response structure."""
    # Coerce all columns to float (Decimal → float); re-cast ints after
    df = df.copy()
    for c in df.columns:
        if c not in ('year_ce', 'n_cells'):
            df[c] = df[c].astype(float)

    result = []
    for _, r in df.iterrows():
        area = r['basin_area_km2'] if r['basin_area_km2'] > 0 else None
        epoch = {
            'year_ce':        int(r['year_ce']),
            'cropland_km2':   r['cropland_km2'],
            'grazing_km2':    r['grazing_km2'],
            'pasture_km2':    r['pasture_km2'],
            'rangeland_km2':  r['rangeland_km2'],
            'basin_area_km2': area,
            'n_cells':        int(r['n_cells']),
        }
        if area:
            for var in ('cropland', 'grazing', 'pasture', 'rangeland'):
                epoch[f'{var}_pct'] = round(r[f'{var}_km2'] / area * 100, 2)
        result.append(epoch)
    return result

rows7 = conn.execute(AGG_SQL, {'hybas_id': hybas_l8, 'from_year': 1000, 'to_year': 1200}).fetchall()
df7 = pd.DataFrame(rows7, columns=cols)
response = rows_to_response(df7)

import json
print(json.dumps(response, indent=2))

In [12]:
# Cell 9 — full aggregation, L6 basin, broad window (0–1900 CE)
# Stress test: many cells × many time steps

AGG_SQL_L6 = AGG_SQL.replace('public.basin08', geom_table_l6)
t0 = time.perf_counter()
rows = conn.execute(AGG_SQL_L6, {'hybas_id': hybas_l6, 'from_year': 0, 'to_year': 1900}).fetchall()
elapsed = time.perf_counter() - t0
df6 = pd.DataFrame(rows, columns=cols)
print(f'L6 broad window (0–1900): {len(df6)} epochs, {elapsed*1000:.0f} ms')
print(df6.to_string(index=False))

L6 broad window (0–1900): 38 epochs, 68 ms
 year_ce cropland_km2 grazing_km2 pasture_km2 rangeland_km2 basin_area_km2  n_cells
       0        0.556      54.297       0.000        54.297         3687.8       45
     100        0.635      61.937       0.000        61.937         3687.8       45
     200        0.726      70.648       0.000        70.648         3687.8       45
     300        0.830      80.581       0.000        80.581         3687.8       45
     400        0.948      91.968       0.000        91.968         3687.8       45
     500        1.084     104.976       0.000       104.976         3687.8       45
     600        1.238     119.830       0.000       119.830         3687.8       45
     700        1.415     136.790       0.000       136.790         3687.8       45
     800        1.618     156.156       0.000       156.156         3687.8       45
     900        1.707     164.598       0.000       164.598         3687.8       45
    1000        1.797     173.011

In [13]:
# Cell 10 — polity polygon proxy query
# Simulates a future Cliopatria polity query: geometry supplied as WKT,
# not a hybas_id join. Use the L6 basin geometry as the proxy polygon.

POLY_SQL = """
WITH basin_cells AS (
    SELECT hc.cropland, hc.grazing, hc.pasture, hc.rangeland, hc.area_km2
    FROM temporal.hyde_cells hc
    WHERE ST_Within(ST_Centroid(hc.geom), ST_GeomFromText(%(wkt)s, 4326))
),
steps AS (
    SELECT step_idx, year_ce
    FROM temporal.hyde_times
    WHERE year_ce BETWEEN %(from_year)s AND %(to_year)s
)
SELECT
    s.year_ce,
    round(SUM(bc.cropland [s.step_idx + 1])::numeric, 3) AS cropland_km2,
    round(SUM(bc.grazing  [s.step_idx + 1])::numeric, 3) AS grazing_km2,
    round(SUM(bc.pasture  [s.step_idx + 1])::numeric, 3) AS pasture_km2,
    round(SUM(bc.rangeland[s.step_idx + 1])::numeric, 3) AS rangeland_km2,
    round(SUM(bc.area_km2)::numeric, 1)                  AS basin_area_km2,
    COUNT(*)                                              AS n_cells
FROM basin_cells bc
CROSS JOIN steps s
GROUP BY s.year_ce
ORDER BY s.year_ce;
"""

wkt = conn.execute(
    f'SELECT ST_AsText(geom) FROM {geom_table_l6} WHERE hybas_id = %s', (hybas_l6,)
).fetchone()[0]
print(f'L6 WKT length: {len(wkt)} chars')

t0 = time.perf_counter()
rows = conn.execute(POLY_SQL, {'wkt': wkt, 'from_year': 0, 'to_year': 1900}).fetchall()
elapsed = time.perf_counter() - t0
df_poly = pd.DataFrame(rows, columns=cols)
print(f'Polygon query (L6 proxy), 0–1900: {len(df_poly)} epochs, {elapsed*1000:.0f} ms')
print(df_poly.to_string(index=False))

L6 WKT length: 24846 chars
Polygon query (L6 proxy), 0–1900: 38 epochs, 30 ms
 year_ce cropland_km2 grazing_km2 pasture_km2 rangeland_km2 basin_area_km2  n_cells
       0        0.556      54.297       0.000        54.297         3687.8       45
     100        0.635      61.937       0.000        61.937         3687.8       45
     200        0.726      70.648       0.000        70.648         3687.8       45
     300        0.830      80.581       0.000        80.581         3687.8       45
     400        0.948      91.968       0.000        91.968         3687.8       45
     500        1.084     104.976       0.000       104.976         3687.8       45
     600        1.238     119.830       0.000       119.830         3687.8       45
     700        1.415     136.790       0.000       136.790         3687.8       45
     800        1.618     156.156       0.000       156.156         3687.8       45
     900        1.707     164.598       0.000       164.598         3687.8       4

In [16]:
# Cell 11 — response shape: what hyde.py will return
# Convert to a list of dicts (one per epoch) and add pct columns.
# This is the structure the API will serialize.

def rows_to_response(df):
    """Convert aggregation result to the API response structure."""
    result = []
    for _, r in df.iterrows():
        area = r['basin_area_km2'] if r['basin_area_km2'] > 0 else None
        epoch = {
            'year_ce':           int(r['year_ce']),
            'cropland_km2':      float(r['cropland_km2']),
            'grazing_km2':       float(r['grazing_km2']),
            'pasture_km2':       float(r['pasture_km2']),
            'rangeland_km2':     float(r['rangeland_km2']),
            'basin_area_km2':    float(area) if area else None,
            'n_cells':           int(r['n_cells']),
        }
        if area:
            for var in ('cropland', 'grazing', 'pasture', 'rangeland'):
                epoch[f'{var}_pct'] = round(float(r[f'{var}_km2']) / area * 100, 2)
        result.append(epoch)
    return result

# Use the L8 narrow-window result from Cell 7
t0 = time.perf_counter()
rows7 = conn.execute(AGG_SQL, {'hybas_id': hybas_l8, 'from_year': 1000, 'to_year': 1200}).fetchall()
df7 = pd.DataFrame(rows7, columns=cols)
response = rows_to_response(df7)

import json
print(json.dumps(response, indent=2))

TypeError: unsupported operand type(s) for /: 'float' and 'decimal.Decimal'

In [18]:
 # fix: raw rows, no pandas, no Decimal issues
import json

rows7 = conn.execute(AGG_SQL, {'hybas_id': hybas_l8, 'from_year': 1000, 'to_year': 1200}).fetchall()

response = []
for row in rows7:
  year_ce, cropland, grazing, pasture, rangeland, area_km2, n_cells = row
  area = float(area_km2) if area_km2 else None
  epoch = {
      'year_ce':        int(year_ce),
      'cropland_km2':   float(cropland),
      'grazing_km2':    float(grazing),
      'pasture_km2':    float(pasture),
      'rangeland_km2':  float(rangeland),
      'basin_area_km2': area,
      'n_cells':        int(n_cells),
  }
  if area:
      for var, val in [('cropland', cropland), ('grazing', grazing),
                       ('pasture', pasture), ('rangeland', rangeland)]:
          epoch[f'{var}_pct'] = round(float(val) / area * 100, 2)
  response.append(epoch)

print(json.dumps(response, indent=2))

[
  {
    "year_ce": 1000,
    "cropland_km2": 0.079,
    "grazing_km2": 4.705,
    "pasture_km2": 0.0,
    "rangeland_km2": 4.705,
    "basin_area_km2": 573.4,
    "n_cells": 7,
    "cropland_pct": 0.01,
    "grazing_pct": 0.82,
    "pasture_pct": 0.0,
    "rangeland_pct": 0.82
  },
  {
    "year_ce": 1100,
    "cropland_km2": 0.085,
    "grazing_km2": 4.905,
    "pasture_km2": 0.0,
    "rangeland_km2": 4.905,
    "basin_area_km2": 573.4,
    "n_cells": 7,
    "cropland_pct": 0.01,
    "grazing_pct": 0.86,
    "pasture_pct": 0.0,
    "rangeland_pct": 0.86
  },
  {
    "year_ce": 1200,
    "cropland_km2": 0.09,
    "grazing_km2": 5.083,
    "pasture_km2": 0.0,
    "rangeland_km2": 5.083,
    "basin_area_km2": 573.4,
    "n_cells": 7,
    "cropland_pct": 0.02,
    "grazing_pct": 0.89,
    "pasture_pct": 0.0,
    "rangeland_pct": 0.89
  }
]


In [19]:
# Cell 12 — summary timing table

results = []

scenarios = [
    ('L8 narrow (1000–1200)',   hybas_l8, 'public.basin08', 1000,   1200),
    ('L8 broad  (0–1900)',      hybas_l8, 'public.basin08', 0,      1900),
    ('L8 full   (-10000–2025)', hybas_l8, 'public.basin08', -10000, 2025),
    ('L6 narrow (1000–1200)',   hybas_l6, geom_table_l6,    1000,   1200),
    ('L6 broad  (0–1900)',      hybas_l6, geom_table_l6,    0,      1900),
    ('L6 full   (-10000–2025)', hybas_l6, geom_table_l6,    -10000, 2025),
]

for label, hid, tbl, fy, ty in scenarios:
    sql = AGG_SQL if tbl == 'public.basin08' else AGG_SQL.replace('public.basin08', tbl)
    t0 = time.perf_counter()
    rows = conn.execute(sql, {'hybas_id': hid, 'from_year': fy, 'to_year': ty}).fetchall()
    ms = (time.perf_counter() - t0) * 1000
    results.append({'scenario': label, 'n_epochs': len(rows), 'ms': round(ms, 0)})

summary = pd.DataFrame(results)
print(summary.to_string(index=False))

               scenario  n_epochs    ms
  L8 narrow (1000–1200)         3 676.0
     L8 broad  (0–1900)        38 642.0
L8 full   (-10000–2025)       128 312.0
  L6 narrow (1000–1200)         3  29.0
     L6 broad  (0–1900)        38  27.0
L6 full   (-10000–2025)       128  39.0


In [ ]:
# Cell 13 — close connection
conn.close()
print('Done.')